# 🏠 House Price Prediction — مشروع متكامل
**Dataset:** [Kaggle - House Data (shree1992/housedata)](https://www.kaggle.com/datasets/shree1992/housedata)

**الخطوات:** تحميل البيانات → EDA → تنظيف → تدريب النماذج → تقييم → حفظ النموذج

## الخطوة 1: تثبيت المكتبات وتحميل البيانات من كاجل على طول


In [ ]:
!pip install kagglehub -q

import kagglehub, os, joblib
path = kagglehub.dataset_download('shree1992/housedata')
print('Dataset path:', path)
print(os.listdir(path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

df = pd.read_csv(os.path.join(path, 'data.csv'))
df.head()


## الخطوة 2: استكشاف البيانات (EDA)


In [ ]:
print('Shape:', df.shape)
df.info()
display(df.describe())

missing = df.isnull().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame('missing'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price'], kde=True, ax=axes[0]).set_title('Price Distribution')
sns.scatterplot(x='sqft_living', y='price', data=df, ax=axes[1]).set_title('Sqft Living vs Price')
plt.show()


In [ ]:
num_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(14, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## الخطوة 3: التنظيف والمعالجة


In [ ]:
df = df.dropna(subset=['price'])

# عمود sqft_above فيه فواصل — نصلّحه
if df['sqft_above'].dtype == object:
    df['sqft_above'] = df['sqft_above'].str.replace(',', '').astype(float)

# حذف أعمدة مش هتفيد
df = df.drop(columns=['date', 'street', 'country'], errors='ignore')

# ترميز الكاتيجوري
from sklearn.preprocessing import LabelEncoder
df['city_encoded'] = LabelEncoder().fit_transform(df['city'].astype('category'))
df['statezip_encoded'] = LabelEncoder().fit_transform(df['statezip'].astype('category'))
df = df.drop(columns=['city', 'statezip'])

# إزالة الـ outliers باستخدام الكوانتايل 1% و 99%
q1, q3 = df['price'].quantile([0.01, 0.99])
df = df[(df['price'] >= q1) & (df['price'] <= q3)]
print('Shape بعد التنظيف:', df.shape)


## الخطوة 4: تقسيم البيانات والتدريب


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
}

results = {}
trained = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_pred = m.predict(X_test_s)
    trained[name] = m
    results[name] = {
        'R2': round(r2_score(y_test, y_pred), 4),
        'MAE': round(mean_absolute_error(y_test, y_pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
    }

results_df = pd.DataFrame(results).T
display(results_df)


## الخطوة 5: أفضل نموذج + رسم الفعلي vs المتوقع + حفظ الملفات


In [ ]:
best_name = results_df['R2'].idxmax()
best_model = trained[best_name]
print('Best model:', best_name)

y_pred = best_model.predict(X_test_s)
plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Actual vs Predicted — {best_name}')
plt.show()


In [ ]:
# حفظ النموذج والـ scaler وأسماء الأعمدة
os.makedirs('model_artifacts', exist_ok=True)
joblib.dump(best_model, 'model_artifacts/house_model.pkl')
joblib.dump(scaler, 'model_artifacts/scaler.pkl')
joblib.dump(X.columns.tolist(), 'model_artifacts/features.pkl')
print('Saved! Files:', os.listdir('model_artifacts'))
